# Experimental-style TCSPC import and estimation

This notebook exercises the single-curve CSV path with a **synthetic stand-in**, not a real experimental measurement. Its generator settings are documented in data/examples/README.md but are not attached as truth to the imported measurement or used during estimation. For a real acquisition, verify the time calibration, detector background, IRF acquisition conditions, and mono-exponential model suitability before interpreting the fit.

In [ ]:
from pathlib import Path
import numpy as np

from tcspc_toolkit.config import FeatureConfig, PreprocessingConfig
from tcspc_toolkit.experimental import (
    evaluate_estimate_against_reference,
    fit_experimental_reconvolution,
    measurement_to_feature_table,
    measurement_to_histogram_batch,
)
from tcspc_toolkit.features import FEATURE_NAMES
from tcspc_toolkit.measurement_io import load_sampled_irf_csv, load_tcspc_measurement_csv
from tcspc_toolkit.ml_models import make_normalized_histogram_ridge_pipeline
from tcspc_toolkit.simulation import simulate_irf_convolved_histogram

candidate = Path.cwd()
repo_root = candidate if (candidate / 'data' / 'examples').is_dir() else candidate.parent
example_dir = repo_root / 'data' / 'examples'
assert example_dir.is_dir()

## Import and validate

Time units and count semantics are declared at import. The histogram file uses picoseconds; the separate sampled IRF uses nanoseconds. Both grids are converted to nanoseconds and checked with the preprocessing time-axis tolerance. The raw source arrays and provenance remain available; no background subtraction, count normalization, or IRF resampling happens on import.

In [ ]:
sampled_irf = load_sampled_irf_csv(
    example_dir / 'experimental_style_irf.csv',
    time_unit='ns',
    time_column='time_ns',
)
measurement = load_tcspc_measurement_csv(
    example_dir / 'experimental_style_decay.csv',
    time_unit='ps',
    data_kind='raw_counts',
    time_column='time_ps',
    irf=sampled_irf,
    sample_id='synthetic-import-demo',
)
print('sample:', measurement.sample_id)
print('kind:', measurement.data_kind.value, 'bins:', measurement.time_ns.size)
print('time window (ns):', measurement.time_ns[0], measurement.time_ns[-1])
print('total raw photons:', measurement.values.sum())
print('histogram file:', Path(measurement.provenance['source_path']).name)
print('histogram provenance:', {k: v for k, v in measurement.provenance.items() if k != 'source_path'})
print('IRF file:', Path(measurement.irf.provenance['source_path']).name)
print('IRF provenance:', {k: v for k, v in measurement.irf.provenance.items() if k != 'source_path'})

## Analysis settings and classical estimate

The existing configuration classes describe optional preprocessing/feature choices. This example keeps the imported raw counts unchanged for Poisson inference. The small local settings dictionary records only the fit choices used here; it is **not** a persistent analysis-lineage system. The measured IRF is normalized only inside the fitting adapter. Temporal-shift bounds are an explicit acquisition-dependent assumption.

In [ ]:
preprocessing_config = PreprocessingConfig()
feature_config = FeatureConfig(tail_start_ns=6.0, early_stop_ns=2.0, late_start_ns=6.0)
analysis_settings = {'temporal_shift_bounds_ns': (-0.5, 0.5), 'background_fraction': 0.10}
print('preprocessing:', preprocessing_config, 'fit settings:', analysis_settings)
result = fit_experimental_reconvolution(measurement, **analysis_settings)
print('valid fit:', result.valid_fit, 'optimizer success:', result.optimizer_success)
print('lifetime (ns):', result.fitted_lifetime_ns)
print('amplitude:', result.fitted_amplitude, 'background/bin:', result.fitted_background)
print('relative IRF shift (ns):', result.fitted_temporal_shift_ns)
print('Poisson deviance:', result.poisson_deviance, 'Poisson NLL:', result.poisson_nll)
print('boundary hit:', result.boundary_hit, 'runtime (ms):', result.runtime_ms)
print('failure reason:', result.failure_reason)
assert result.valid_fit

The scalar Poisson deviance is a fit diagnostic, **not** a universal test of physical model adequacy. A narrow statistical interval or a small scalar diagnostic can coexist with a wrong decay model. The imported measurement has no intrinsic reference lifetime, so no MAE, bias, or coverage is computed.

## ML input on a compatible grid — demonstration only

The feature adapter uses the existing feature schema and the raw histogram adapter uses the existing normalized-histogram pipeline. Below, a tiny model is fitted **only on separately generated synthetic development curves**. It is not fitted, selected, calibrated, or evaluated on the imported curve or frozen A–F benchmark. Its prediction on the imported file is an unvalidated transfer demonstration, not evidence of synthetic-to-real accuracy.

In [ ]:
feature_row = measurement_to_feature_table(
    measurement,
    training_time_ns=measurement.time_ns,
    feature_config=feature_config,
    training_feature_names=FEATURE_NAMES,
)
print('engineered feature columns:', tuple(feature_row.columns))

rng = np.random.default_rng(7)
development_lifetimes_ns = np.linspace(1.0, 6.0, 40)
development_histograms = np.vstack([
    simulate_irf_convolved_histogram(
        time=measurement.time_ns,
        lifetime_ns=float(lifetime_ns),
        signal_photon_count=5000,
        background_per_bin=2.0,
        irf_centre_ns=1.0,
        irf_fwhm_ns=0.6,
        irf_shift_ns=0.0,
        rng=rng,
    )[0]
    for lifetime_ns in development_lifetimes_ns
])
model = make_normalized_histogram_ridge_pipeline()
model.fit(development_histograms, development_lifetimes_ns)
experimental_input = measurement_to_histogram_batch(
    measurement, training_time_ns=measurement.time_ns
)
ml_estimate_ns = float(model.predict(experimental_input)[0])
print('unvalidated synthetic-to-import ML estimate (ns):', ml_estimate_ns)

## Optional trusted-reference evaluation

An experimental curve has no built-in generating lifetime. Only call the separate reference evaluator if an independently justified reference value and its provenance are available. Its signed error is **estimate minus reference**. A single reference comparison is not a validation study and cannot establish bias or coverage.

In [ ]:
trusted_reference_ns = None
if trusted_reference_ns is not None:
    comparison = evaluate_estimate_against_reference(
        result.fitted_lifetime_ns,
        reference_lifetime_ns=trusted_reference_ns,
        reference_metadata={'source': 'describe the independent calibration here'},
    )
    print('signed reference error (ns):', comparison.signed_error_ns)
else:
    print('No trusted reference supplied; reference-based metrics omitted.')